### Create inference and pkl for the training feature

In [1]:
!pip install --upgrade pip

In [19]:
import pandas as pd
import sklearn
import joblib
import boto3
import sagemaker
import google.protobuf

print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("boto3:", boto3.__version__)
print("sagemaker:", sagemaker.__version__)
print("protobuf:", google.protobuf.__version__)

pandas: 2.2.3
scikit-learn: 1.6.1
joblib: 1.5.0
boto3: 1.37.1
sagemaker: 2.244.0
protobuf: 5.28.3


In [20]:
# Import Libraries
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import pickle
import tarfile
import joblib
import boto3
import os
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import Session
import sagemaker

In [21]:
# Load the dataset
df = pd.read_csv("s3://sagemaker-us-east-1-993768311527/cardio_data/cardio_prod_split40.csv")
print(df.head())

        age  height_ft  weight_lbs  systolic_bp  diastolic_bp  cholesterol  \
0 -0.860483  -0.323580   -0.914107    -0.410657     -0.147087    -0.538657   
1 -0.268438   1.211180    0.266123    -0.410657     -0.147087    -0.538657   
2  0.915651  -1.052592   -0.983384    -0.410657     -0.147087    -0.538657   
3 -0.712472  -0.822378   -1.191845    -0.410657     -0.147087    -0.538657   
4  0.323606  -0.170104   -0.358631    -0.410657     -0.147087    -0.538657   

       gluc     smoke     alco    active  ...  age_years  is_hypertensive  \
0 -0.390761 -0.312731 -0.23822  0.494625  ...  -0.860483        -0.607947   
1 -0.390761 -0.312731 -0.23822 -2.021734  ...  -0.268438        -0.607947   
2 -0.390761 -0.312731 -0.23822  0.494625  ...   0.915651        -0.607947   
3 -0.390761 -0.312731 -0.23822  0.494625  ...  -0.712472        -0.607947   
4 -0.390761 -0.312731 -0.23822  0.494625  ...   0.323606        -0.607947   

   age_gluc_interaction  lifestyle_score  gender  bp_category  bmi_c

In [22]:
# Generate_model_and_inference.py

import pandas as pd
import joblib
import os
import tarfile
from sklearn.linear_model import LogisticRegression

# === Step 1: Load and prepare data ===
s3_path = "s3://sagemaker-us-east-1-993768311527/cardio_data/cardio_prod_split40.csv"
df = pd.read_csv(s3_path)

# Save inference data without label (no header)
df.drop(columns=["cardio"]).to_csv("cardio_prod_no_label.csv", index=False, header=False)

# Split into X and y
X = df.drop(columns=["cardio"])
y = df["cardio"]

# === Step 2: Train the model ===
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# Save model as .joblib
os.makedirs("model", exist_ok=True)
model_path = "model/logistic_model.joblib"
joblib.dump(model, model_path, protocol=4)


# === Step 3: Create inference.py file ===
inference_code = '''
import pandas as pd
from io import StringIO
import joblib
import os

FEATURE_COLUMNS = [
    'age', 'gender', 'height_ft', 'weight_lbs', 'systolic_bp', 'diastolic_bp',
    'cholesterol', 'gluc', 'smoke', 'alco', 'active',
    'bmi', 'age_group', 'cholesterol_label', 'pulse_pressure', 'chol_bmi_ratio',
    'height_in', 'age_years', 'is_hypertensive', 'bp_category', 'bmi_category',
    'age_gluc_interaction', 'lifestyle_score'
]

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "logistic_model.joblib"))

def input_fn(input_data, content_type):
    if content_type == "text/csv":
        df = pd.read_csv(StringIO(input_data), header=None)
        if df.shape[1] != len(FEATURE_COLUMNS):
            raise ValueError(f"Column mismatch: expected {len(FEATURE_COLUMNS)}, got {df.shape[1]}")
        df.columns = FEATURE_COLUMNS
        return df
    else:
        raise ValueError(f"Unsupported content type: {content_type}")

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, content_type):
    return '\\n'.join(str(x) for x in prediction)
'''

with open("model/inference.py", "w") as f:
    f.write(inference_code)

# === Step 4: Package model and code for SageMaker ===
with tarfile.open("logistic_model.tar.gz", "w:gz") as tar:
    tar.add("model/logistic_model.joblib", arcname="logistic_model.joblib")
    tar.add("model/inference.py", arcname="inference.py")

print("✅ logistic_model.tar.gz created successfully.")

✅ logistic_model.tar.gz created successfully.


### Save files directly into S3 Bucket

In [23]:
# Define your bucket and prefix
bucket = 'sagemaker-us-east-1-993768311527' # Prema's new Path
prefix = 'model'

# Your filenames (assuming you created these earlier)
model_filename = 'model/logistic_model.joblib'
tar_filename = 'logistic_model.tar.gz'
inference_csv_file = 'cardio_prod_no_label.csv'
inference_file = 'model/inference.py'

# Upload all files to S3 (overwrite automatically)
s3_client = boto3.client('s3')

s3_client.upload_file(model_filename, bucket, f"{prefix}/logistic_model.joblib")
s3_client.upload_file(inference_file, bucket, f"{prefix}/inference.py")
s3_client.upload_file(tar_filename, bucket, f"{prefix}/{tar_filename}")
s3_client.upload_file(inference_csv_file, bucket, f"{prefix}/{inference_csv_file}")

print("All files uploaded successfully!")

All files uploaded successfully!


### Compare Features in Training vs Inference Data

In [24]:
# Load the training data (with label)
# df_train = pd.read_csv("s3://sagemaker-us-east-1-381492296191/cardio_data/cardio_prod_split40.csv") # Prema's old Path
df_train = pd.read_csv("s3://sagemaker-us-east-1-993768311527/cardio_data/cardio_prod_split40.csv") # Prema's new path
train_features = df_train.drop(columns=["cardio"]).columns.tolist()

# Load the inference data (no label)
df_infer = pd.read_csv("cardio_prod_no_label.csv", header=None)

# Compare number of columns
print("Training Features Count:", len(train_features))
print("Inference Features Count:", df_infer.shape[1])

# Set column names on inference data to match training features for manual inspection (Optional)
df_infer.columns = train_features

# Compare data types and column names
print("\nTraining Feature Types:")
print(df_train[train_features].dtypes)

print("\nInference Data Types:")
print(df_infer.dtypes)

# Identify mismatched columns (by type or order)
mismatch_columns = [
    (col, df_train[col].dtype, df_infer[col].dtype)
    for col in train_features
    if df_train[col].dtype != df_infer[col].dtype
]

if mismatch_columns:
    print("\nMismatched Columns Found:")
    for col, train_type, infer_type in mismatch_columns:
        print(f"   - {col}: training type = {train_type}, inference type = {infer_type}")
else:
    print("\nAll feature columns match in name and dtype.")

Training Features Count: 23
Inference Features Count: 23

Training Feature Types:
age                     float64
height_ft               float64
weight_lbs              float64
systolic_bp             float64
diastolic_bp            float64
cholesterol             float64
gluc                    float64
smoke                   float64
alco                    float64
active                  float64
bmi                     float64
pulse_pressure          float64
chol_bmi_ratio          float64
height_in               float64
age_years               float64
is_hypertensive         float64
age_gluc_interaction    float64
lifestyle_score         float64
gender                  float64
bp_category             float64
bmi_category            float64
age_group               float64
cholesterol_label       float64
dtype: object

Inference Data Types:
age                     float64
height_ft               float64
weight_lbs              float64
systolic_bp             float64
diastolic_bp     

### Creating inference.py file

In [25]:
with tarfile.open("logistic_model.tar.gz", "w:gz") as tar:
    tar.add("model/logistic_model.joblib", arcname="logistic_model.joblib")
    tar.add("model/inference.py", arcname="inference.py")

### Sagemaker Inference Transform Job

In [26]:
# Get SageMaker session and role
import boto3
from sagemaker import Session
from sagemaker.sklearn.model import SKLearnModel

# Set up session and variables
sagemaker_session = Session()
role = sagemaker.get_execution_role()
bucket = "sagemaker-us-east-1-993768311527"
prefix = "model"
model_artifact = f"s3://{bucket}/{prefix}/logistic_model.tar.gz"
inference_input = f"s3://{bucket}/{prefix}/cardio_prod_no_label.csv"
output_path = f"s3://{bucket}/{prefix}/inference-output"

# Create Model object
sklearn_model = SKLearnModel(
    model_data=model_artifact,
    role=role,
    entry_point="inference.py",
    source_dir="./model",  # must point to where inference.py is during packaging
    framework_version="0.23-1",
    sagemaker_session=sagemaker_session
)

# Batch Transform job
transformer = sklearn_model.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    strategy="SingleRecord",
    output_path=output_path,
    assemble_with="Line",
    accept="text/csv"
)

# Launch Transform Job
transformer.transform(
    data=inference_input,
    content_type="text/csv",
    split_type="Line",
    input_filter="$[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22]",  # assuming 23 features
    join_source="Input",
    wait=True
)

print("✅ Inference transform job completed.")

INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2025-06-19-04-14-54-999
INFO:sagemaker:Creating transform job with name: sagemaker-scikit-learn-2025-06-19-04-14-56-103


...........................2025-06-19 04:19:28,081 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2025-06-19 04:19:28,085 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)
2025-06-19 04:19:28,086 INFO - sagemaker-containers - nginx config: 
worker_processes auto;
daemon off;
pid /tmp/nginx.pid;
error_log  /dev/stderr;
worker_rlimit_nofile 4096;
events {
  worker_connections 2048;
}
http {
  include /etc/nginx/mime.types;
  default_type application/octet-stream;
  access_log /dev/stdout combined;
  upstream gunicorn {
    server unix:/tmp/gunicorn.sock;
  }
  server {
    listen 8080 deferred;
    client_max_body_size 0;
    keepalive_timeout 3;
    location ~ ^/(ping|invocations|execution-parameters) {
      proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
      proxy_set_header Host $http_host;
      proxy_redirect off;
      proxy_read_timeout 60s;
      proxy_pass http://gunicorn;
    }
    location / {
      retur

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:36                                                                                   │
│                                                                                                  │
│   33 )                                                                                           │
│   34                                                                                             │
│   35 # Launch Transform Job                                                                      │
│ ❱ 36 transformer.transform(                                                                      │
│   37 │   data=inference_input,                                                                   │
│   38 │   content_type="text/csv",                                                                │
│   39 │   split_type="Line",                                                                      │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/transformer.py:318 in transform                │
│                                                                                                  │
│   315 │   │   )                                                                                  │
│   316 │   │                                                                                      │
│   317 │   │   if wait:                                                                           │
│ ❱ 318 │   │   │   self.latest_transform_job.wait(logs=logs)                                      │
│   319 │                                                                                          │
│   320 │   def transform_with_monitoring(                                                         │
│   321 │   │   self,                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/transformer.py:686 in wait                     │
│                                                                                                  │
│   683 │                                                                                          │
│   684 │   def wait(self, logs=True):                                                             │
│   685 │   │   if logs:                                                                           │
│ ❱ 686 │   │   │   self.sagemaker_session.logs_for_transform_job(self.job_name, wait=True)        │
│   687 │   │   else:                                                                              │
│   688 │   │   │   self.sagemaker_session.wait_for_transform_job(self.job_name)                   │
│   689                                                      

In [32]:
!aws s3 cp cardio_inference_transform_job_v2.ipynb s3://sagemaker-us-east-1-531690656306/cardio_project/cardio_inference_transform_job_v2.ipynb

upload: ./cardio_inference_transform_job_v2.ipynb to s3://sagemaker-us-east-1-531690656306/cardio_project/cardio_inference_transform_job_v2.ipynb


In [33]:
# Save all files to s#
s3 = boto3.client('s3')
bucket = 'sagemaker-us-east-1-993768311527'
prefix = 'cardio_data/'  # or whatever path you use

files_to_upload = [
    'logistic_model.tar.gz',
    'logistic_model.pkl',
    'inference.py',
    'cardio_prod_no_label.csv'
]

for file in files_to_upload:
    s3.upload_file(file, bucket, prefix + file)
    print(f"Uploaded: {file} → s3://{bucket}/{prefix}{file}")

Uploaded: logistic_model.tar.gz → s3://sagemaker-us-east-1-531690656306/model/logistic_model.tar.gz
Uploaded: logistic_model.pkl → s3://sagemaker-us-east-1-531690656306/model/logistic_model.pkl
Uploaded: inference.py → s3://sagemaker-us-east-1-531690656306/model/inference.py
Uploaded: cardio_prod_no_label.csv → s3://sagemaker-us-east-1-531690656306/model/cardio_prod_no_label.csv


In [35]:
# Display save files
# List objects in the specified S3 prefix
s3 = boto3.client("s3")
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

# Extract and display filenames
s3_files = [obj["Key"] for obj in response.get("Contents", [])]
s3_files

['model/',
 'model/cardio_prod_no_label.csv',
 'model/inference.py',
 'model/logistic_model.pkl',
 'model/logistic_model.tar.gz']